# Task 3 · Car Price Prediction with Machine Learning
**Objective:** predict the selling price of a used car from brand, age, mileage (km driven), fuel type and transmission.

**Dataset:** Kaggle → *Vehicle dataset from CarDekho* → file `CAR DETAILS FROM CAR DEKHO.csv`. Put it in the same folder as this notebook.

## 1. Imports

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_style('whitegrid')

## 2. Load data

In [ ]:
df = pd.read_csv('CAR DETAILS FROM CAR DEKHO.csv')   # change the file name if yours differs
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe(include='all').T

## 3. Data cleaning
Handle nulls, duplicates and inconsistent categorical values (e.g. `Petrol` vs `petrol`).

In [ ]:
print('Nulls:\n', df.isnull().sum())
df = df.dropna()
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# standardise text columns: strip spaces + consistent case
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()
for col in ['fuel', 'seller_type', 'transmission', 'owner']:
    if col in df.columns:
        df[col] = df[col].str.title()
        print(col, '->', df[col].unique())
print(df.shape)

## 4. Feature engineering
- **car_age** = current year − manufacturing year
- **brand** = first word of the car name

In [ ]:
CURRENT_YEAR = 2026
df['car_age'] = CURRENT_YEAR - df['year']
df['brand'] = df['name'].str.split().str[0].str.title()
df['brand'] = df['brand'].replace({'Land': 'Land Rover', 'Maruti': 'Maruti'})  # 'Land Rover' splits into 'Land'
print(df['brand'].value_counts().head(10))
# group rare brands so one-hot encoding stays manageable
top = df['brand'].value_counts()
df['brand'] = df['brand'].where(df['brand'].isin(top[top >= 20].index), 'Other')
df = df.drop(columns=['name', 'year'])
df.head()

## 5. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['selling_price'], bins=50, kde=True, ax=ax[0]); ax[0].set_title('Selling price distribution')
sns.histplot(np.log1p(df['selling_price']), bins=50, kde=True, ax=ax[1]); ax[1].set_title('log(Selling price)')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='fuel', y='selling_price', data=df)
plt.yscale('log'); plt.title('Price vs fuel type'); plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='car_age', y='selling_price', data=df, alpha=0.4)
plt.yscale('log'); plt.title('Price vs car age'); plt.show()

**Observations:** prices are heavily right-skewed (log scale is more symmetric); older cars sell for much less; diesel/petrol medians differ visibly.

## 6. Encode categorical variables (One-Hot Encoding)

In [ ]:
# log-transform target to tame the skew; we convert predictions back to rupees when evaluating
y = np.log1p(df['selling_price'])
X = pd.get_dummies(df.drop(columns=['selling_price']), drop_first=True)
X.head()

## 7. Correlation heatmap

In [ ]:
corr = X.assign(selling_price=df['selling_price']).corr()
plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Feature correlation heatmap'); plt.show()
print(corr['selling_price'].sort_values(ascending=False).head(8))

## 8. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 9. Train models (Linear Regression, Random Forest, Gradient Boosting)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, random_state=42),
}
for m in models.values():
    m.fit(X_train, y_train)

## 10. Evaluate (MAE, RMSE, R²) — in original price units

In [ ]:
rows = []
y_true = np.expm1(y_test)
for name, m in models.items():
    pred = np.expm1(m.predict(X_test))
    rows.append({'Model': name,
                 'MAE': mean_absolute_error(y_true, pred),
                 'RMSE': np.sqrt(mean_squared_error(y_true, pred)),
                 'R2': r2_score(y_true, pred)})
results = pd.DataFrame(rows).set_index('Model').round(3)
results

In [ ]:
best_name = results['R2'].idxmax()
print('Best model:', best_name)
best = models[best_name]
pred = np.expm1(best.predict(X_test))
plt.figure(figsize=(6,6))
plt.scatter(y_true, pred, alpha=0.4)
lim = [0, max(y_true.max(), pred.max())]
plt.plot(lim, lim, 'r--'); plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title(f'{best_name}: actual vs predicted'); plt.show()

## 11. Feature importance (best model)

In [ ]:
if hasattr(best, 'feature_importances_'):
    imp = pd.Series(best.feature_importances_, index=X.columns).sort_values().tail(15)
else:
    imp = pd.Series(np.abs(best.coef_), index=X.columns).sort_values().tail(15)
imp.plot(kind='barh', figsize=(8,6)); plt.title(f'Top features - {best_name}'); plt.show()

## 12. Conclusion
- Best model: *(fill in from the results table)* with R² ≈ *(value)*.
- Most important drivers: car age, km driven, fuel type, transmission and brand.
- Possible improvements: hyper-parameter tuning, cross-validation, outlier handling.